# DATT Phase 3.2: Full Pipeline Execution on Google Colab CUDA (Tesla T4)

This notebook demonstrates and validates the end-to-end **DATT - AI People Counter** pipeline running on a **Google Colab Tesla T4 GPU** with PyTorch CUDA acceleration.

### Pipeline Architecture:
```
YouTube Live (yt-dlp) -> FFmpeg (bgr24 raw pipe) -> CameraReader (zero-queue thread)
  -> YOLODetector (YOLO11s PyTorch CUDA) -> PersonTracker (ByteTrack) -> ZoneCounter -> Realtime HUD Logging
```

### Runtime Requirements:
- Menu: `Runtime` -> `Change runtime type` -> Hardware accelerator: **T4 GPU**

In [1]:
import torch

# Step 1: Verify CUDA GPU availability
cuda_available = torch.cuda.is_available()
print(f"CUDA available = {cuda_available}")

if cuda_available:
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Device Count: {torch.cuda.device_count()} | Compute Capability: {torch.cuda.get_device_capability(0)}")
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024 ** 2)
    alloc_vram = torch.cuda.memory_allocated(0) / (1024 ** 2)
    print(f"Total VRAM: {total_vram:.2f} MB | Allocated VRAM: {alloc_vram:.2f} MB")
else:
    print("WARNING: Running on CPU. Please switch Colab runtime to GPU (T4).")

CUDA available = True
Device Name: Tesla T4
Device Count: 1 | Compute Capability: (7, 5)
Total VRAM: 15102.00 MB | Allocated VRAM: 0.00 MB


In [2]:
# Step 2: Install minimal Colab dependencies
!pip install -q -r requirements-colab.txt

In [3]:
import os
import sys

# Ensure repo root is on sys.path
repo_root = os.path.abspath(".")
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import config
from src.detector.yolo_detector import YOLODetector
from src.tracker.bytetrack_tracker import PersonTracker
from src.counter.zone_counter import ZoneCounter
from src.stream.youtube_stream import CameraReader

In [4]:
# Step 3: Initialize Detector and check model parameters
detector = YOLODetector(config)
print(f"Detector initialized successfully on: {detector.device} ({detector.device_type}, {detector.device_name})")
print(f"Model: {detector.model_path.name}")
print(f"Input Size: {config.IMG_SIZE}")

Detector initialized successfully on: cuda:0 (CUDA, Tesla T4)
Model: yolo11s.pt
Input Size: 640


In [5]:
# Step 4: Run full pipeline on live YouTube stream for 100 frames
from app import run_pipeline

results = run_pipeline(max_frames=100)

print("\n--- BENCHMARK RESULTS (100 frames on Tesla T4) ---")
print(f"Average YOLO Latency:     {results['avg_yolo_ms']:.1f} ms")
print(f"Average Pipeline Latency: {results['avg_pipeline_ms']:.1f} ms")
print(f"Processing FPS:           {results['processing_fps']:.1f} FPS")
print(f"Stream FPS:               {results['stream_fps']:.1f} FPS")
print(f"VRAM Allocated:           {results['vram_mb']:.1f} MB")

Starting DATT - AI People Counter (Phase 3.2 Colab CUDA)
[INFO] Starting camera stream from YouTube Live...

==================== [REALTIME HUD] ====================
Device:           CUDA
GPU:              Tesla T4
VRAM:             248.5 MB
Stream FPS:       30.2
Processing FPS:   68.5
YOLO latency:     12.1 ms
Pipeline latency: 14.3 ms
Detection count:  3
Track count:      3
People in view:   3

==================== [REALTIME HUD] ====================
Device:           CUDA
GPU:              Tesla T4
VRAM:             248.5 MB
Stream FPS:       30.0
Processing FPS:   71.4
YOLO latency:     11.8 ms
Pipeline latency: 13.9 ms
Detection count:  3
Track count:      3
People in view:   3

Reached maximum requested frames (100). Stopping.
Pipeline stopped cleanly.

--- BENCHMARK RESULTS (100 frames on Tesla T4) ---
Average YOLO Latency:     12.2 ms
Average Pipeline Latency: 14.5 ms
Processing FPS:           69.8 FPS
Stream FPS:               30.0 FPS
VRAM Allocated:           248.5 MB
